In [4]:
# Frozen MambaVision future-latent prediction on Waymo E2E

Architecture:
FRONT camera images → frozen NVIDIA MambaVision-T2-1K → cached pooled features
→ context concatenation → residual MLP predictor → future frozen-encoder features.

The same frozen encoder produces context and target features. Only the predictor
is trained. There is no additional Temporal Mamba, image decoder, trajectory
planner, or vehicle-control head.

This is a JEPA-style latent prediction prototype, not end-to-end JEPA:
the encoder does not learn from the prediction objective, and there is no EMA
teacher or joint representation learning.

Data limitation:
E2EDFrame.future_states contains vehicle positions, not future images.
Future visual targets must come from independently verified chronological records.
Do not treat TFRecord order, filename order, or context.name as proof of sequence
membership. The official E2E schema describes context.name as a frame identifier.

Requirements:
- Colab GPU runtime, preferably T4.
- Authorized local copies of Waymo E2E TFRecords.
- A CSV manifest linking records to independently verified sequences.
- Training and validation sequences that do not overlap.
- Enough consecutive images at the configured sampling interval.

Evaluation:
Compare against persistence: predict that every future latent equals the latest
context latent. Report per-horizon standardized MSE and cosine similarity.
These are representation metrics, not driving safety or planning metrics.

Sources:
- https://github.com/waymo-research/waymo-open-dataset/blob/master/src/waymo_open_dataset/protos/end_to_end_driving_data.proto
- https://huggingface.co/nvidia/MambaVision-T2-1K

SyntaxError: invalid character '→' (U+2192) (2789077658.py, line 4)

In [1]:

import torch
import os

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: enable a T4 GPU in Colab!")


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [3]:

# Waymo protobufs; avoid replacing Colab's TensorFlow stack
%pip -q install --no-deps waymo-open-dataset-tf-2-12-0==1.6.7

# NVIDIA MambaVision and feature extraction libraries
%pip -q install mambavision "transformers>=4.40,<5" timm pillow



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 12.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 10.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 40.8 MB/s eta 0:00

In [ ]:
import struct
import google_crc32c
import pandas as pd
from PIL import Image
from io import BytesIO
from collections import Counter

def masked_crc32c(data):
    crc = google_crc32c.value(data)
    return (((crc >> 15) | (crc << 17)) + 0xA282EAD8) & 0xFFFFFFFF

def iter_tfrecord(path):
    """Read uncompressed TFRecords, checking both CRCs and record bounds."""
    with open(path, "rb") as handle:
        index = 0
        while True:
            header = handle.read(12)
            if not header:
                return
            if len(header) != 12:
                raise ValueError(f"Truncated TFRecord header: {path}")
            length_bytes, length_crc = header[:8], header[8:]
            length = struct.unpack("<Q", length_bytes)[0]

            if struct.unpack("<I", length_crc)[0] != masked_crc32c(length_bytes):
                raise ValueError(f"TFRecord header CRC failure: {path}:{index}")
            if length > 256 * 1024 * 1024:
                raise ValueError(f"Unreasonable record size: {length}")

            payload = handle.read(length)
            payload_crc = handle.read(4)
            if len(payload) != length or len(payload_crc) != 4:
                raise ValueError(f"Truncated record: {path}:{index}")
            if struct.unpack("<I", payload_crc)[0] != masked_crc32c(payload):
                raise ValueError(f"TFRecord payload CRC failure: {path}:{index}")

            yield index, payload
            index += 1

def parse_e2e(payload):
    message = e2e_pb2.E2EDFrame()
    message.ParseFromString(payload)

    if not message.HasField("frame"):
        raise ValueError("Missing E2EDFrame.frame")
    frame = message.frame

    if not frame.HasField("timestamp_micros") or frame.timestamp_micros <= 0:
        raise ValueError("Missing/invalid frame timestamp")
    if not frame.HasField("context") or not frame.context.name:
        raise ValueError("Missing frame identifier")

    cameras = [im for im in frame.images if im.name == FRONT]
    if len(cameras) != 1 or not cameras[0].image:
        raise ValueError("Expected exactly one nonempty FRONT image")

    image = Image.open(BytesIO(cameras[0].image))
    if image.format != "JPEG":
        raise ValueError("FRONT camera payload is not JPEG")
    image.load()
    image = image.convert("RGB")

    # Optional trajectory visualization only. Never used as future image targets.
    xy = None
    if message.HasField("future_states"):
        xs = list(message.future_states.pos_x)
        ys = list(message.future_states.pos_y)
        if len(xs) != len(ys):
            raise ValueError("Unequal future trajectory coordinate lengths")
        if xs:
            xy = np.asarray([xs, ys], dtype=np.float32).T
            if not np.isfinite(xy).all():
                raise ValueError("Nonfinite future trajectory")

    return image, int(frame.timestamp_micros), frame.context.name, xy

def load_manifest(path):
    if not path.exists():
        raise FileNotFoundError(
            f"Create {path} from verified sequence metadata. "
            "The notebook will not infer temporal sequences."
        )

    table = pd.read_csv(path, dtype={
        "sequence_id": str, "split": str, "tfrecord": str,
    })
    required = {
        "sequence_id", "split", "tfrecord",
        "record_index", "timestamp_micros",
    }
    if not required.issubset(table.columns):
        raise ValueError(f"Required manifest columns: {sorted(required)}")
    if table.empty or table[list(required)].isna().any().any():
        raise ValueError("Empty manifest or missing required values")

    for column in ["record_index", "timestamp_micros"]:
        numeric = pd.to_numeric(table[column], errors="raise")
        if not np.isfinite(numeric).all() or (numeric % 1 != 0).any():
            raise ValueError(f"{column} must contain integers")
        table[column] = numeric.astype("int64")

    if (table.record_index < 0).any() or (table.timestamp_micros <= 0).any():
        raise ValueError("Invalid record indices/timestamps")
    if set(table.split) != {"train", "val"}:
        raise ValueError("Provide both train and val, and no other splits")
    if table.groupby("sequence_id").split.nunique().max() != 1:
        raise ValueError("Sequence leakage between train and validation")
    if table.duplicated(["tfrecord", "record_index"]).any():
        raise ValueError("Duplicate record in manifest")
    if table.duplicated(["sequence_id", "timestamp_micros"]).any():
        raise ValueError("Duplicate timestamp within sequence")

    def resolve_path(value):
        p = Path(value).expanduser()
        return str((path.parent / p).resolve() if not p.is_absolute()
                   else p.resolve())

    table["tfrecord"] = table.tfrecord.map(resolve_path)
    for filename in table.tfrecord.unique():
        if not Path(filename).is_file():
            raise FileNotFoundError(filename)

    # Bound the prototype without mixing train/validation.
    table = table.sort_values(["split", "sequence_id", "timestamp_micros"])
    table = table.groupby("split", sort=False).head(
        CFG["max_frames_per_split"]
    ).reset_index(drop=True)
    table["row_id"] = np.arange(len(table))
    return table

manifest = load_manifest(MANIFEST)
display(manifest.head())
print(manifest.groupby("split").size())